In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import accelerate
import torch
import pandas as pd
import numpy as np
os.environ["HUGGINGFACE_TOKEN"] = "INSERT TOKEN"
cache_dir=' '
#model_name = "microsoft/phi-4"
#model_name="google/gemma-2-27b-it"
#model_name="meta-llama/Llama-3.2-3B-Instruct"
model_name="meta-llama/Llama-3.1-70B-Instruct"
#model_name="CohereLabs/aya-expanse-8b"
#model_name="tiiuae/falcon-7b-instruct"
#model_name="mistralai/Mixtral-8x7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto", use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)


#model_name = "tiiuae/Falcon3-Mamba-7B-Instruct"

#model = AutoModelForCausalLM.from_pretrained(
#    model_name,
#    torch_dtype=torch.bfloat16,
#    device_map="auto",
# cache_dir=cache_dir
#)
#tokenizer = AutoTokenizer.from_pretrained(model_name)


/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/tokenization_auto.py:823: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

In [2]:
import pandas as pd
import numpy as np
file_path="cambridgev2.csv"
df=pd.read_csv(file_path)
df=df.drop(["Unnamed: 0"],axis=1)
#df=df.iloc[:,:3]
df.head()

,ID,Text,Label,LLAMA_70b_vanilla,LLAMA_70b_avg,LLAMA_70b_entropy,LLAMA_8b_vanilla,LLAMA_8b_avg,LLAMA_8b_entropy,LLAMA_3b_vanilla,...,conf_noscale_aya_32b_entropy,conf_noscale_aya_32b_verbal_conf,conf_noscale_mixtral_vanilla,conf_noscale_mixtral_avg,conf_noscale_mixtral_entropy,conf_noscale_mixtral_verbal_conf,conf_mixtral_vanilla,conf_mixtral_avg,conf_mixtral_entropy,conf_mixtral_verbal_conf
0,152,"Blacksmiths\n\nTHROUGHOUT the ages, iron has e...",4,3,3.000000,-0.000000,4,3.841130316257477,0.037221,4,...,0.000377,7.952573180198669,2,2.0019266445888206,0.001347,8.001649048324907,3,3.565413,0.074787,8.002838
1,236,The two sisters kept Lily's driving a secret f...,5,4,4.000000,-0.000000,4,4.0,-0.000000,4,...,0.000028,7.622458080863709,3,2.9241418246752984,0.025887,8.001642447008635,5,5.011483,0.010519,7.880797
2,28,"Family Business\n\n'Look here, it's no good!' ...",5,3,2.889273,0.029590,4,3.6513550877571106,0.054975,4,...,0.051924,7.977022799037826,2,2.000552669633829,0.000453,8.017985758517653,3,2.983008,0.018566,8.002043
3,137,Where I get my energy\n\nEmma Marsden asked si...,3,2,2.000000,-0.000000,2,2.1329642087221146,0.033327,3,...,0.000728,7.9914212273433805,3,2.5621765915326447,0.066072,8.005189024748688,5,4.901173,0.028554,7.999678
4,210,Photography\n\nWhen a photographer takes a pho...,5,4,3.602684,0.057126,4,3.7391715943813324,0.048795,4,...,0.000004,7.777299756369963,2,2.0002034196216,0.000186,8.002134083590136,5,4.539347,0.081167,7.982014


In [24]:
df.columns

Index(['ID', 'Text', 'Label', 'LLAMA_70b_vanilla', 'LLAMA_70b_avg',
       'LLAMA_70b_entropy', 'LLAMA_8b_vanilla', 'LLAMA_8b_avg',
       'LLAMA_8b_entropy', 'LLAMA_3b_vanilla', 'LLAMA_3b_avg',
       'LLAMA_3b_entropy', 'gemma_27b_vanilla', 'gemma_27b_avg',
       'gemma_27b_entropy', 'gemma_2b_vanilla', 'gemma_2b_avg',
       'gemma_2b_entropy', 'gemma_9b_vanilla', 'gemma_9b_avg',
       'gemma_9b_entropy', 'phi_vanilla', 'phi_avg', 'phi_entropy',
       'AYA_32B_vanilla', 'AYA_32B_avg', 'AYA_32B_entropy', 'AYA_8B_vanilla',
       'AYA_8B_avg', 'AYA_8B_entropy', 'Falcon_vanilla', 'Falcon_avg',
       'Falcon_entropy', 'mixtral_vanilla', 'mixtral_avg', 'mixtral_entropy',
       'noscale_mixtral_vanilla', 'noscale_mixtral_avg',
       'noscale_mixtral_entropy', 'noscale_aya_32b_vanilla',
       'noscale_aya_32b_avg', 'noscale_aya_32b_entropy',
       'noscale_aya_8b_vanilla', 'noscale_aya_8b_avg',
       'noscale_aya_8b_entropy', 'noscale_llama_8b_vanilla',
       'noscale_llama_8b_av

In [3]:
## Gets readability scores without eliciting model confidence

## LLAMA 3B or 8B or 70B or Mixtral or Aya 8B or 32B or phi 4 or gemma 2b/9b/27B
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,1]
    score=df.iloc[i,2]
        ##CEFR PROMPT
    #prompt=f"Rate the readability of the text between 1 (easy) and 5 (very challenging) using the following scale: 1 = Can understand short, simple texts on familiar matters of a concrete type 2 = Can read straightforward factual texts on subjects related to his/her field and interest with a satisfactory level of comprehension. 3 = Can read with a large degree of independence, adapting style and speed of reading to different texts and purpose 4 = Can understand in detail lengthy, complex texts, whether or not they relate to his/her own area of speciality, provided he/she can reread difficult sections. 5 = Can understand and interpret critically virtually all forms of the written language including abstract, structurally complex, or highly colloquial literary and non-literary writings. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Only answer in this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    
        ## NO SCALE PROMPT
    prompt=f"Rate the readability of each passage with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"
  
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=4,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-1][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',tokenizer.decode(outputs.sequences[0][-25:]))
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens,top6_probs.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
    real_scores.append(score)
    entropies.append(normalized_entropy)
        #try:
        #    count+=int(tokenizer.decode(outputs[0][-1]))
        #    count1+=1
        #except ValueError:
        #    try:
            #print(tokenizer.decode(outputs[0][-4:]),int(tokenizer.decode(outputs[0][-1])))
        #        count+=int(tokenizer.decode(outputs[0][-2]))
         #       count1+=1
         #   except ValueError:
         #       print('error')
         #       print(tokenizer.decode(outputs[0][-5:]))
   # try:
   #     scores_llm.append(count/count1)
   #     scores.append(score)
   # except ValueError:
   #     print('Big error')
   # except ZeroDivisionError:
   #     print('Zero error')
   #     scores_llm.append('na')
   #     scores.append('na')

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
df['noscale_llama_70b_vanilla']=scores_llm_vanilla
df['noscale_llama_70b_avg']=scores_llm_avg
df['noscale_llama_70b_entropy']=entropies
#print(corrs)
df.to_csv("cambridgev2.csv")

From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


ERROR - FIRST TOKEN NOT NUM  with this format: Answer: [SCORE] Explanation: [EXPLANATION]<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Answer: Citizen Kane
ERROR - FIRST TOKEN NOT NUM  with this format: Answer: [SCORE] Explanation: [EXPLANATION]<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here are the readability
ERROR - FIRST TOKEN NOT NUM  with this format: Answer: [SCORE] Explanation: [EXPLANATION]<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here are the readability
ERROR - FIRST TOKEN NOT NUM  with this format: Answer: [SCORE] Explanation: [EXPLANATION]<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here are the readability
ERROR - FIRST TOKEN NOT NUM  with this format: Answer: [SCORE] Explanation: [EXPLANATION]<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here are the readability
ERROR - FIRST TOKEN NOT NUM  with this format: Answer: [SCORE] Explanation: [EXPLANATION]<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Her

In [33]:
## Gets readability scores and confidence scores

corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
verbal_confs=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,1]
    score=df.iloc[i,2]
        ##CEFR PROMPT
    prompt = f"Rate the readability of the text between 1 (easy) and 5 (very challenging) using the following scale: 1 = Can understand short, simple texts on familiar matters of a concrete type 2 = Can read straightforward factual texts on subjects related to his/her field and interest with a satisfactory level of comprehension. 3 = Can read with a large degree of independence, adapting style and speed of reading to different texts and purpose 4 = Can understand in detail lengthy, complex texts, whether or not they relate to his/her own area of speciality, provided he/she can reread difficult sections. 5 = Can understand and interpret critically virtually all forms of the written language including abstract, structurally complex, or highly colloquial literary and non-literary writings. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Answer with this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    
        ## NO SCALE PROMPT
    #prompt=f"Rate the readability of the text with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Answer with this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=10,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    logits_last=scores[-3][0]
    logits_fifth_to_last = scores[-7][0]
    import torch
    probs_last = torch.softmax(logits_last, dim=-1)
    probs_fifth_to_last = torch.softmax(logits_fifth_to_last, dim=-1)
    entropy = -torch.sum(probs_fifth_to_last * torch.log(probs_fifth_to_last + 1e-12)).item()
    vocab_size = probs_fifth_to_last.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
    top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())

    top6_probs_fifth_to_last, top6_ids_fifth_to_last = torch.topk(probs_fifth_to_last, 6)
    top6_tokens_fifth_to_last=tokenizer.convert_ids_to_tokens(top6_ids_fifth_to_last.tolist())

    
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens_fifth_to_last[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM')
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
        
    avg2=0
    flag2=False
    try:
        vanilla_conf=int(top6_tokens_last[0])
    except ValueError:
        try:
            logits_last=scores[-2][0]
            probs_last = torch.softmax(logits_last, dim=-1)
            top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
            top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
            vanilla_conf=int(top6_tokens_last[0])
        except ValueError:
            try:
                logits_last=scores[-1][0]
                probs_last = torch.softmax(logits_last, dim=-1)
                top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
                top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
                vanilla_conf=int(top6_tokens_last[0])
            except ValueError:
                
                print('ERROR - FIRST CONFIDENCE TOKEN NOT NUM',tokenizer.decode(outputs['sequences'][0][-10:]))
                flag2=True
                verbal_confs.append('na')
    if flag2==False:
        for (n,p) in zip(top6_tokens_last,top6_probs_last.tolist()):
            try:
                avg2+=int(n)*p
            except ValueError:
                #print(avg)
                break
        verbal_confs.append(avg2)
    real_scores.append(score)
    entropies.append(normalized_entropy)

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
filtered_entropies, filtered_verbal_confs = zip(*[(e,v) for e,v in zip(entropies, verbal_confs) if e != 'na' and v != 'na'])
print(np.corrcoef(filtered_entropies, filtered_verbal_confs)[0, 1])

df['conf_mixtral_vanilla']=scores_llm_vanilla
df['conf_mixtral_avg']=scores_llm_avg
df['conf_mixtral_entropy']=entropies
df['conf_mixtral_verbal_conf']=verbal_confs
#print(corrs)
df.to_csv("cambridgev2.csv")

0.7609106315974665 300 0.7794194591514125 300
-0.3884817719349728


In [12]:
df

,ID,Text,Label,LLAMA_70b_vanilla,LLAMA_70b_avg,LLAMA_70b_entropy,LLAMA_8b_vanilla,LLAMA_8b_avg,LLAMA_8b_entropy,LLAMA_3b_vanilla,...,conf_noscale_phi_entropy,conf_noscale_phi_verbal_conf,conf_noscale_gemma_27b_vanilla,conf_noscale_gemma_27b_avg,conf_noscale_gemma_27b_entropy,conf_noscale_gemma_27b_verbal_conf,conf_noscale_gemma_2b_vanilla,conf_noscale_gemma_2b_avg,conf_noscale_gemma_2b_entropy,conf_noscale_gemma_2b_verbal_conf
0,152,"Blacksmiths\n\nTHROUGHOUT the ages, iron has e...",4,3,3.000000,-0.000000,4,3.841130316257477,0.037221,4,...,0.098293,7.777390,5,4.524529,0.067032,6.961318,6,6.022867,0.018530,7.290832
1,236,The two sisters kept Lily's driving a secret f...,5,4,4.000000,-0.000000,4,4.0,-0.000000,4,...,0.066511,7.473475,4,4.155642,0.044203,6.83724,6,5.918359,0.029601,7.31438
2,28,"Family Business\n\n'Look here, it's no good!' ...",5,3,2.889273,0.029590,4,3.6513550877571106,0.054975,4,...,0.081295,7.905598,4,3.570008,0.061556,6.990527,6,5.870146,0.038111,7.216144
3,137,Where I get my energy\n\nEmma Marsden asked si...,3,2,2.000000,-0.000000,2,2.1329642087221146,0.033327,3,...,0.060514,7.976612,3,3.050229,0.032126,7.101037,6,5.69062,0.062166,7.061007
4,210,Photography\n\nWhen a photographer takes a pho...,5,4,3.602684,0.057126,4,3.7391715943813324,0.048795,4,...,0.114952,7.038416,5,4.825641,0.068560,6.777978,na,na,0.008720,7.29437
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,126,Trills and bills \n\nIf there's one thing guar...,3,2,2.000000,-0.000000,2,2.2608282268047333,0.048795,3,...,0.063074,7.949186,3,3.210865,0.050574,7.013447,na,na,0.021465,6.681527
296,226,TIM RICE\n\nI was ushered into the young man's...,5,3,3.091815,0.026078,4,3.841131180524826,0.037221,4,...,0.097758,7.882981,4,4.076305,0.040548,6.935538,6,5.963295,0.026447,7.102584
297,67,The Ghan Train \n\nMark Ottaway travelled by t...,2,2,2.000000,-0.000000,3,3.1887218952178955,0.041181,4,...,0.066468,7.958923,4,3.941359,0.037575,6.990691,6,6.007976,0.030485,7.293462
298,25,Art on TV\n\nWhy is it that television so cons...,5,3,2.889273,0.029590,3,3.3029409050941467,0.052146,3,...,0.081010,7.947954,5,4.929528,0.072528,6.83684,na,na,0.012774,7.348596


In [8]:
##FALCON 
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,1]
    score=df.iloc[i,2]

    prompt=f"Rate the readability of the text between 1 (easy) and 5 (very challenging) using the following scale: 1 = Can understand short, simple texts on familiar matters of a concrete type 2 = Can read straightforward factual texts on subjects related to his/her field and interest with a satisfactory level of comprehension. 3 = Can read with a large degree of independence, adapting style and speed of reading to different texts and purpose 4 = Can understand in detail lengthy, complex texts, whether or not they relate to his/her own area of speciality, provided he/she can reread difficult sections. 5 = Can understand and interpret critically virtually all forms of the written language including abstract, structurally complex, or highly colloquial literary and non-literary writings. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Only answer in this format: Answer: [SCORE] Explanation: [EXPLANATION]"
   
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    outputs = model.generate(**model_inputs, max_new_tokens=4,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-1][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',top6_tokens)
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens,top6_probs.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
    real_scores.append(score)
    entropies.append(normalized_entropy)

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
df['Falcon_vanilla']=scores_llm_vanilla
df['Falcon_avg']=scores_llm_avg
df['Falcon_entropy']=entropies
#print(corrs)
df.to_csv("cambridgev2.csv")

ERROR - FIRST TOKEN NOT NUM ['IST', 'OC', 'IGN', '.', 'ESS', 'UMP']
ERROR - FIRST TOKEN NOT NUM ['RE', 'RES', 'PE', 'UR', 'OR', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'UR', 'RES', 'OR', 'LE']
ERROR - FIRST TOKEN NOT NUM ["'", 'Ġget', 'Ġthink', 'Ġdon', 'Ġhave', 'Ġlove']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'RES', 'UR', 'OR', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'UR', 'OR', 'RES', 'LE']
ERROR - FIRST TOKEN NOT NUM ['IST', 'OC', 'IGN', '.', 'Ġ', 'ESS']
ERROR - FIRST TOKEN NOT NUM ['RE', 'OR', 'UR', 'PE', 'RES', 'LE']
ERROR - FIRST TOKEN NOT NUM ['IST', 'OC', '.', 'IGN', "'", 'Ġ']
ERROR - FIRST TOKEN NOT NUM ['RE', 'UR', 'RES', 'PE', 'OR', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'RES', 'UR', 'OR', 'LE']
ERROR - FIRST TOKEN NOT NUM ['Ġa', 'Ġan', 'Ġabout', 'Ġfrom', 'Ġtaken', 'Ġthe']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'UR', 'RES', 'OR', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'OR', 'UR', 'RES', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'UR', 'PE', 'RES

/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/numpy/lib/function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/numpy/lib/function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/numpy/lib/function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
